# TP 4 — Fédération de données multi-SGBD avec PySpark

**Big Data Engineering — Master 1 Data Science — ISI / UCAD**
Prof. S. Ndiaye — DMI / FST / UCAD

---

## 🎯 Objectifs

Au TP 3, Spark a joint deux tables issues de **deux bases MySQL** distinctes.
Aujourd'hui, on franchit un cap : joindre **quatre tables** hébergées dans
**quatre SGBD différents** — MySQL, PostgreSQL, Oracle XE et SQL Server
Express — comme si elles appartenaient à une seule base. C'est la
**fédération de données** : aucun SGBD ne sait faire cela seul, Spark si.

À la fin de ce TP, vous saurez :

1. embarquer **plusieurs pilotes JDBC** dans une même `SparkSession` ;
2. lire des tables depuis 4 SGBD aux dialectes différents ;
3. construire une **vue fédérée** par jointures multi-sources ;
4. auditer l'**intégrité référentielle** inter-SGBD (anti-jointures) ;
5. écrire le résultat consolidé dans un entrepôt (PostgreSQL).

## 📋 Consignes

- Complétez les cellules marquées `# === À COMPLÉTER ===` (remplacez les `...`).
- Exécutez le notebook **de bout en bout** (Noyau → Redémarrer et tout exécuter).
- Poussez le notebook **avec ses sorties** sur votre dépôt GitHub avant la séance 5.


---
# 0. Vérification de l'environnement

Comme aux TP 2 et 3 : on affiche l'interpréteur actif, on répare PySpark si
le noyau sélectionné ne le voit pas, et on vérifie Java (**17 recommandé**).

In [45]:
import sys, os

print("Interpreteur Python actif :", sys.executable)
print("Version                   :", sys.version.split()[0])

Interpreteur Python actif : C:\Users\SERIGNE MBACKE GUEYE\AppData\Local\Programs\Python\Python311\python.exe
Version                   : 3.11.9


In [46]:
# Auto-reparation : installe PySpark 3.5.1 dans le noyau ACTIF si absent
try:
    import pyspark
    print("PySpark", pyspark.__version__, "est deja disponible.")
except ModuleNotFoundError:
    print("PySpark introuvable dans ce noyau -> installation...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "pyspark==3.5.1"])
    import pyspark
    print("PySpark", pyspark.__version__, "installe.")

PySpark 3.5.1 est deja disponible.


In [47]:
# Verification de Java (indispensable a Spark)
import shutil, subprocess

java = shutil.which("java")
if java is None:
    raise EnvironmentError(
        "Java est introuvable dans le PATH.\n"
        "-> Installez un JDK 17 (Adoptium Temurin), puis definissez JAVA_HOME\n"
        "   et redemarrez Jupyter depuis un terminal ou 'java -version' repond.")
res = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(res.stderr.strip() or res.stdout.strip())

openjdk version "17.0.20" 2026-07-21
OpenJDK Runtime Environment Temurin-17.0.20+8 (build 17.0.20+8)
OpenJDK 64-Bit Server VM Temurin-17.0.20+8 (build 17.0.20+8, mixed mode, sharing)


In [48]:
# Forcer Spark a utiliser CE python (evite les erreurs de workers Windows)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print("Interpreteur impose a Spark :", sys.executable)

Interpreteur impose a Spark : C:\Users\SERIGNE MBACKE GUEYE\AppData\Local\Programs\Python\Python311\python.exe


In [49]:
# Tableau de releves : completez-le au fil du TP (comme aux TP 2 et 3)
RELEVES = {}

---
# 1. Le contexte : quatre équipes, quatre SGBD

Notre plateforme e-commerce sénégalaise a grandi. Chaque équipe a choisi
**son** SGBD, pour de bonnes (et de moins bonnes) raisons historiques :

| Équipe | Table | SGBD | Port | Base / schéma |
|---|---|---|---|---|
| CRM | `clients` (12) | **MySQL 8** | 3306 | `ecommerce_crm` |
| Catalogue | `produits` (10) | **PostgreSQL 16** | 5432 | `ecommerce_catalogue` |
| Ventes | `commandes` (30) | **Oracle XE 21c** | 1521 | schéma `SPARK_USER` |
| Finance | `paiements` (27) | **SQL Server 2022 Express** | 1433 | `ecommerce_paiements` |

Il n'existe **aucune clé étrangère** entre ces systèmes : l'intégrité
référentielle inter-SGBD n'est garantie par personne. La direction demande
pourtant un chiffre d'affaires consolidé par ville, par catégorie de produit
et par moyen de paiement. **Spark est le seul outil de la pile qui parle
aux quatre à la fois.**

```
   MySQL         PostgreSQL       Oracle XE       SQL Server
  (clients)      (produits)      (commandes)     (paiements)
     │               │                │               │
     └───────┬───────┴───────┬────────┴───────┬───────┘
             │    4 pilotes JDBC dans la JVM  │
             └──────────► SparkSession ◄──────┘
                              │
                    vue fédérée + analyses
                              │
                     PostgreSQL (analytics)
```


## 1.1 Paramètres de connexion

**Une seule cellule à adapter** si vos mots de passe ou ports diffèrent.
Le compte `spark_user` / `Ucad2026!` a été créé par les 4 scripts
`init_tp4_*.sql` (partie A du sujet).

In [63]:
HOTE = "localhost"

UTILISATEUR = "spark_user"
import os
MDP = os.environ.get("TP_MDP", "")

# --- MySQL (clients) ---
MYSQL_URL    = (f"jdbc:mysql://{HOTE}:3306/ecommerce_crm"
                "?useSSL=false&allowPublicKeyRetrieval=true")
MYSQL_PILOTE = "com.mysql.cj.jdbc.Driver"

# --- PostgreSQL (produits) ---
PG_URL    = f"jdbc:postgresql://{HOTE}:5432/ecommerce_catalogue"
PG_PILOTE = "org.postgresql.Driver"

# --- Oracle XE (commandes) ---
ORACLE_URL    = f"jdbc:oracle:thin:@//{HOTE}:1521/FREEPDB1"
ORACLE_PILOTE = "oracle.jdbc.OracleDriver"

# --- SQL Server Express (paiements) ---
MSSQL_URL    = (f"jdbc:sqlserver://{HOTE}:1433;"
                "databaseName=ecommerce_paiements;"
                "encrypt=true;trustServerCertificate=true")
MSSQL_PILOTE = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

print("Parametres charges pour les 4 SGBD.")

Parametres charges pour les 4 SGBD.


---
# 2. Pré-vérification : les 4 SGBD écoutent-ils ?

Avant de lancer Spark, on teste les 4 ports TCP. **Un port fermé maintenant
= une erreur JDBC illisible dans 10 minutes.** Chaque échec affiche la
marche à suivre pour le SGBD concerné.

> ⚠️ **Piège n°1 du TP** : SQL Server Express est installé avec le
> protocole **TCP/IP désactivé** par défaut ! Voir le message d'aide.

In [51]:
import socket

SGBD = [
    ("MySQL",      3306, "Demarrez le service : services.msc -> MySQL80 -> Demarrer\n"
                         "  (ou : net start MySQL80 dans un terminal administrateur)"),
    ("PostgreSQL", 5432, "Demarrez le service : services.msc -> postgresql-x64-16 -> Demarrer"),
    ("Oracle XE",  1521, "Demarrez les services OracleServiceXE ET OracleOraDB21Home1TNSListener\n"
                         "  (services.msc). Le listener DOIT etre demarre AVANT la base."),
    ("SQL Server", 1433, "1) SQL Server Configuration Manager -> Protocoles pour SQLEXPRESS\n"
                         "     -> TCP/IP -> Activer ; onglet Adresses IP -> IPAll ->\n"
                         "     Port TCP = 1433 (et vider 'Ports dynamiques TCP')\n"
                         "  2) Redemarrer le service SQL Server (SQLEXPRESS)\n"
                         "  3) Verifier que l'authentification en MODE MIXTE est activee"),
]

tout_va_bien = True
for nom, port, aide in SGBD:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    ok = (s.connect_ex((HOTE, port)) == 0)
    s.close()
    print(("[ OK  ] " if ok else "[ECHEC] ") + f"{nom:<11} port {port}")
    if not ok:
        tout_va_bien = False
        print("        -> " + aide + "\n")

if not tout_va_bien:
    raise ConnectionError(
        "Au moins un SGBD ne repond pas : corrigez avant de continuer.")
print("\nLes 4 SGBD ecoutent : on peut continuer !")

[ OK  ] MySQL       port 3306
[ OK  ] PostgreSQL  port 5432
[ OK  ] Oracle XE   port 1521
[ OK  ] SQL Server  port 1433

Les 4 SGBD ecoutent : on peut continuer !


---
# 3. Les quatre pilotes JDBC

Au TP 3, un seul `.jar` suffisait. Ici, la JVM de Spark doit embarquer
**quatre** pilotes — un par dialecte :

| SGBD | Artefact Maven | Classe du pilote |
|---|---|---|
| MySQL | `mysql-connector-j` 8.4.0 | `com.mysql.cj.jdbc.Driver` |
| PostgreSQL | `postgresql` 42.7.3 | `org.postgresql.Driver` |
| Oracle | `ojdbc11` 23.4 | `oracle.jdbc.OracleDriver` |
| SQL Server | `mssql-jdbc` 12.6.1.jre11 | `com.microsoft.sqlserver.jdbc.SQLServerDriver` |

La cellule ci-dessous télécharge les 4 jars dans `jars/` (avec le contrôle
de taille anti-fichier-corrompu du TP 3). **Si la salle n'a pas Internet**,
copiez les jars fournis par l'enseignant dans le dossier `jars/` : la
cellule les détectera et ne téléchargera rien.

In [52]:
from pathlib import Path
from urllib.request import urlretrieve

MAVEN = "https://repo1.maven.org/maven2"
JARS = {
    "mysql-connector-j-8.4.0.jar":
        f"{MAVEN}/com/mysql/mysql-connector-j/8.4.0/mysql-connector-j-8.4.0.jar",
    "postgresql-42.7.3.jar":
        f"{MAVEN}/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar",
    "ojdbc11-23.4.0.24.05.jar":
        f"{MAVEN}/com/oracle/database/jdbc/ojdbc11/23.4.0.24.05/ojdbc11-23.4.0.24.05.jar",
    "mssql-jdbc-12.6.1.jre11.jar":
        f"{MAVEN}/com/microsoft/sqlserver/mssql-jdbc/12.6.1.jre11/mssql-jdbc-12.6.1.jre11.jar",
}
TAILLE_MIN = 500_000  # octets : en-deca, le fichier est un debris de telechargement

dossier = Path("jars")
dossier.mkdir(exist_ok=True)
chemins = []
for nom, url in JARS.items():
    cible = dossier / nom
    if cible.exists() and cible.stat().st_size < TAILLE_MIN:
        print(f"[!] {nom} corrompu ({cible.stat().st_size} o) -> suppression")
        cible.unlink()
    if not cible.exists():
        print(f"Telechargement de {nom}...")
        try:
            urlretrieve(url, cible)
        except Exception as e:
            raise ConnectionError(
                f"Echec du telechargement de {nom}.\n"
                f"-> Telechargez-le manuellement : {url}\n"
                f"-> puis placez-le dans le dossier {dossier.resolve()}") from e
    print(f"[ OK ] {nom} ({cible.stat().st_size:,} octets)")
    chemins.append(str(cible.resolve()))

LISTE_JARS = ",".join(chemins)   # spark.jars attend une liste separee par des virgules
print("\nspark.jars =", LISTE_JARS[:80] + "...")

[ OK ] mysql-connector-j-8.4.0.jar (2,533,399 octets)
[ OK ] postgresql-42.7.3.jar (1,089,312 octets)
[ OK ] ojdbc11-23.4.0.24.05.jar (7,170,864 octets)
[ OK ] mssql-jdbc-12.6.1.jre11.jar (1,470,328 octets)

spark.jars = C:\Boum\venv_TP\jars\mysql-connector-j-8.4.0.jar,C:\Boum\venv_TP\jars\postgresql...


---
# 4. Une SparkSession, quatre pilotes

Rappel du TP 3 : `spark.jars` se fixe **à la création** de la session.
Nouveauté : on lui passe les **4 jars d'un coup** (chaîne séparée par des
virgules). Si vous ajoutez un jar plus tard, redémarrez le noyau.

In [53]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("TP4 - Federation multi-SGBD - UCAD")
         .master("local[*]")
         .config("spark.jars", LISTE_JARS)
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession creee !")
print("Version de Spark :", spark.version)
print("Interface web    : http://localhost:4040")

SparkSession creee !
Version de Spark : 3.5.1
Interface web    : http://localhost:4040


---
# 5. Lire les quatre tables

## 💡 Ce qu'il faut comprendre

Le gabarit JDBC est **identique** pour les 4 SGBD — seuls changent l'URL,
le nom de table et la classe du pilote. C'est toute la beauté de JDBC :
Spark ne voit que des DataFrames, les dialectes restent cachés derrière
les pilotes.

On factorise donc immédiatement (réflexe des TP 2 et 3) :

In [54]:
def lire_jdbc(url, table, pilote, utilisateur=None, mdp=None):
    """Lit une table d'un SGBD quelconque et renvoie un DataFrame Spark."""
    return (spark.read.format("jdbc")
            .option("url", url)
            .option("dbtable", table)
            .option("user", utilisateur or UTILISATEUR)
            .option("password", mdp or MDP)
            .option("driver", pilote)      # obligatoire : sinon "No suitable driver"
            .load())

print("Fonction lire_jdbc definie.")

Fonction lire_jdbc definie.


## 5.1 MySQL → `clients` (guidé)

Première source, entièrement guidée — observez que le schéma vient
**directement** du SGBD (types `date` garantis, sans `inferSchema`).

In [59]:
df_clients = lire_jdbc(MYSQL_URL, "clients", MYSQL_PILOTE)

df_clients.show(5)
df_clients.printSchema()

RELEVES["nb_clients (MySQL)"] = df_clients.count()
print("Nombre de clients :", RELEVES["nb_clients (MySQL)"])
# Observation attendue : 12 clients, villes du Senegal, 2 emails NULL

+---------+-------------+-----------+--------------------+------------+----------------+
|client_id|          nom|      ville|               email|   telephone|date_inscription|
+---------+-------------+-----------+--------------------+------------+----------------+
|     C001|     Awa Diop|      Dakar|  awa.diop@gmail.com|77 123 45 67|      2025-01-15|
|     C002|Moussa Ndiaye|      Thies|moussa.ndiaye@yah...|78 234 56 78|      2025-02-03|
|     C003|   Fatou Sarr|      Dakar|                NULL|76 345 67 89|      2025-02-20|
|     C004|Ibrahima Fall|Saint-Louis|ibrahima.fall@gma...|70 456 78 90|      2025-03-11|
|     C005|   Aminata Ba| Ziguinchor|aminata.ba@outloo...|75 567 89 01|      2025-04-02|
+---------+-------------+-----------+--------------------+------------+----------------+
only showing top 5 rows

root
 |-- client_id: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telephone: str

## 5.2 À vous : PostgreSQL → `produits`

**Exercice 1.** Lisez la table `produits` de la base PostgreSQL
(`PG_URL`, `PG_PILOTE`), affichez-la, et relevez son nombre de lignes.

In [61]:
# === À COMPLÉTER ===
df_produits = lire_jdbc(PG_URL, "produits", PG_PILOTE)

df_produits.show()
df_produits.printSchema()

RELEVES["nb_produits (PostgreSQL)"] = df_produits.count()
print("Nombre de produits :", RELEVES["nb_produits (PostgreSQL)"])

+----------+--------------------+--------------+---------+-----+
|produit_id|                 nom|     categorie|prix_fcfa|stock|
+----------+--------------------+--------------+---------+-----+
|      P001|Smartphone Tecno ...|    Telephonie|    85000|   40|
|      P002|Casque Bluetooth JBL|    Telephonie|    15000|  120|
|      P003|Ordinateur portab...|  Informatique|   350000|   15|
|      P004|       Cle USB 64 Go|  Informatique|     8000|  200|
|      P005|        Boubou brode|          Mode|    45000|   30|
|      P006|   Paire de sneakers|          Mode|    28000|   60|
|      P007|Ventilateur sur pied|Electromenager|    32000|   25|
|      P008|        Mixeur 500 W|Electromenager|    22000|   35|
|      P009|         Parfum Musc|        Beaute|    12000|   80|
|      P010|     Creme au karite|        Beaute|     6000|  150|
+----------+--------------------+--------------+---------+-----+

root
 |-- produit_id: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- ca

## 5.3 À vous : Oracle XE → `commandes`

**Exercice 2.** Lisez la table `commandes` d'Oracle.

> ⚠️ **Piège n°2 (dialecte Oracle)** : la table vit dans le **schéma**
> `SPARK_USER` — indiquez `"spark_user.commandes"` comme nom de table.
> Autre particularité : Oracle stocke les entiers en `NUMBER`, que Spark
> ramène en `decimal(38,10)`. Comparez le schéma obtenu avec celui de
> PostgreSQL : même donnée, dialecte différent !

In [64]:
# === À COMPLÉTER ===
df_commandes = lire_jdbc(ORACLE_URL, "commandes", ORACLE_PILOTE)

df_commandes.show(5)
df_commandes.printSchema()

RELEVES["nb_commandes (Oracle)"] = df_commandes.count()
print("Nombre de commandes :", RELEVES["nb_commandes (Oracle)"])

+-----------+---------+----------+--------+------------+-------------------+------+
|COMMANDE_ID|CLIENT_ID|PRODUIT_ID|QUANTITE|MONTANT_FCFA|      DATE_COMMANDE|STATUT|
+-----------+---------+----------+--------+------------+-------------------+------+
|      CMD01|     C001|      P001|       1|       85000|2026-01-05 00:00:00|livree|
|      CMD02|     C002|      P003|       1|      350000|2026-01-08 00:00:00|livree|
|      CMD03|     C003|      P009|       2|       24000|2026-01-10 00:00:00|livree|
|      CMD04|     C004|      P005|       1|       45000|2026-01-12 00:00:00|livree|
|      CMD05|     C001|      P002|       2|       30000|2026-01-15 00:00:00|livree|
+-----------+---------+----------+--------+------------+-------------------+------+
only showing top 5 rows

root
 |-- COMMANDE_ID: string (nullable = true)
 |-- CLIENT_ID: string (nullable = true)
 |-- PRODUIT_ID: string (nullable = true)
 |-- QUANTITE: decimal(5,0) (nullable = true)
 |-- MONTANT_FCFA: decimal(12,0) (nullable

### Normaliser le dialecte Oracle

Deux différences à gommer tout de suite pour que la suite du TP soit
uniforme : les **noms de colonnes en MAJUSCULES** et les `decimal(38,10)`.

In [ ]:
df_commandes = (df_commandes
    .toDF(*[c.lower() for c in df_commandes.columns])      # colonnes en minuscules
    .withColumn("quantite",     F.col("quantite").cast("int"))
    .withColumn("montant_fcfa", F.col("montant_fcfa").cast("long")))

df_commandes.printSchema()
# Observation attendue : memes types que si la table venait de PostgreSQL

## 5.4 À vous : SQL Server → `paiements`

**Exercice 3.** Lisez la table `paiements` de SQL Server
(table `dbo.paiements`) et relevez son nombre de lignes.

In [65]:
# === À COMPLÉTER ===
df_paiements = lire_jdbc(MSSQL_URL, "paiements", MSSQL_PILOTE)

df_paiements.show(5)
df_paiements.printSchema()

RELEVES["nb_paiements (SQL Server)"] = df_paiements.count()
print("Nombre de paiements :", RELEVES["nb_paiements (SQL Server)"])

+-----------+-----------+-----------------+------------+---------------+-------------+
|paiement_id|commande_id|   moyen_paiement|montant_fcfa|statut_paiement|date_paiement|
+-----------+-----------+-----------------+------------+---------------+-------------+
|      PAY01|      CMD01|     Orange Money|       85000|         reussi|   2026-01-06|
|      PAY02|      CMD02|             Wave|      350000|         reussi|   2026-01-09|
|      PAY03|      CMD03|especes_livraison|       24000|         reussi|   2026-01-11|
|      PAY04|      CMD04|       Free Money|       45000|         reussi|   2026-01-13|
|      PAY05|      CMD05|     Orange Money|       30000|         reussi|   2026-01-16|
+-----------+-----------+-----------------+------------+---------------+-------------+
only showing top 5 rows

root
 |-- paiement_id: string (nullable = true)
 |-- commande_id: string (nullable = true)
 |-- moyen_paiement: string (nullable = true)
 |-- montant_fcfa: integer (nullable = true)
 |-- statu

### 📸 Point d'étape

Quatre DataFrames, quatre SGBD, un seul moteur. Vérifions :

In [66]:
for nom, df in [("clients (MySQL)", df_clients),
                ("produits (PostgreSQL)", df_produits),
                ("commandes (Oracle)", df_commandes),
                ("paiements (SQL Server)", df_paiements)]:
    print(f"{nom:<25} {df.count():>4} lignes   {len(df.columns)} colonnes")

clients (MySQL)             12 lignes   6 colonnes
produits (PostgreSQL)       10 lignes   5 colonnes
commandes (Oracle)          30 lignes   7 colonnes
paiements (SQL Server)      27 lignes   6 colonnes


---
# 6. La vue fédérée : joindre les quatre tables

## 💡 Ce qu'il faut comprendre

Une fois les DataFrames chargés, **leur origine ne compte plus** : Spark
joint une table Oracle et une table MySQL exactement comme deux CSV.
On expose d'abord chaque DataFrame comme **vue temporaire**, puis on écrit
du SQL standard par-dessus — c'est le *lingua franca* de la fédération.

Le modèle logique (implicite, car aucune FK inter-SGBD n'existe) :

```
clients ──< commandes >── produits
                 │
                 ^
             paiements
```


**Exercice 4.** Créez les quatre vues temporaires `clients`,
`produits`, `commandes`, `paiements`.

In [67]:
# === À COMPLÉTER ===
df_clients.createOrReplaceTempView("clients")
df_produits.createOrReplaceTempView("produits")
df_commandes.createOrReplaceTempView("commandes")
df_paiements.createOrReplaceTempView("paiements")

print(spark.catalog.listTables())

[Table(name='clients', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='commandes', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='paiements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='produits', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


## 6.1 Jointure à deux tables (Oracle + MySQL) — guidé

On refait le geste du TP 3, mais **entre deux SGBD différents** :

In [68]:
deux_sgbd = spark.sql("""
    SELECT co.commande_id, cl.nom, cl.ville, co.montant_fcfa, co.statut
    FROM commandes co
    JOIN clients   cl ON co.client_id = cl.client_id
""")
deux_sgbd.show(5)

RELEVES["jointure commandes-clients"] = deux_sgbd.count()
print("Lignes :", RELEVES["jointure commandes-clients"])
# Observation attendue : 29 lignes (30 commandes - 1 client inconnu C999)

+-----------+-------------+-----+------------+------+
|commande_id|          nom|ville|montant_fcfa|statut|
+-----------+-------------+-----+------------+------+
|      CMD01|     Awa Diop|Dakar|       85000|livree|
|      CMD05|     Awa Diop|Dakar|       30000|livree|
|      CMD20|     Awa Diop|Dakar|       32000|livree|
|      CMD02|Moussa Ndiaye|Thies|      350000|livree|
|      CMD14|Moussa Ndiaye|Thies|       12000|livree|
+-----------+-------------+-----+------------+------+
only showing top 5 rows

Lignes : 29


## 6.2 À vous : trois tables (+ PostgreSQL)

**Exercice 5.** Étendez la jointure aux produits : commande, client, ville,
nom du produit, catégorie, montant, statut. Relevez le nombre de lignes et
expliquez la différence avec 6.1 dans la cellule de réflexion.

In [69]:
# === À COMPLÉTER ===
trois_sgbd = spark.sql("""
    SELECT co.commande_id, cl.nom, cl.ville,
           pr.nom AS produit, pr.categorie,
           co.montant_fcfa, co.statut
    FROM commandes co
    JOIN clients  cl ON co.client_id = cl.client_id
    JOIN produits pr ON co.produit_id = pr.produit_id
""")
trois_sgbd.show(5)

RELEVES["jointure 3 tables"] = trois_sgbd.count()
print("Lignes :", RELEVES["jointure 3 tables"])

+-----------+-------------+--------+--------------------+--------------+------------+-------+
|commande_id|          nom|   ville|             produit|     categorie|montant_fcfa| statut|
+-----------+-------------+--------+--------------------+--------------+------------+-------+
|      CMD07|  Ousmane Sow| Kaolack|Ventilateur sur pied|Electromenager|       32000|annulee|
|      CMD20|     Awa Diop|   Dakar|Ventilateur sur pied|Electromenager|       32000| livree|
|      CMD13| Coumba Cisse|   Dakar|Ordinateur portab...|  Informatique|      350000| livree|
|      CMD02|Moussa Ndiaye|   Thies|Ordinateur portab...|  Informatique|      350000| livree|
|      CMD24|Serigne Mbaye|Diourbel|     Creme au karite|        Beaute|        6000| livree|
+-----------+-------------+--------+--------------------+--------------+------------+-------+
only showing top 5 rows

Lignes : 28


## 6.3 À vous : la vue fédérée complète (4 SGBD !)

**Exercice 6.** Ajoutez les paiements **réussis** : la table finale
`ventes_federees` contient, pour chaque commande **livrée et payée avec
succès**, la commande, le client, sa ville, le produit, sa catégorie, le
moyen de paiement et le montant.

Filtres : `co.statut = 'livree'` **et** `pa.statut_paiement = 'reussi'`.

In [72]:
# === À COMPLÉTER ===
ventes_federees = spark.sql("""
    SELECT co.commande_id, cl.nom, cl.ville,
           pr.nom AS produit, pr.categorie,
           pa.moyen_paiement, co.montant_fcfa
    FROM commandes co
    JOIN clients   cl ON co.client_id = cl.client_id
    JOIN produits  pr ON co.produit_id = pr.produit_id
    JOIN paiements pa ON pa.commande_id = co.commande_id
    WHERE co.statut = 'livree' AND pa.statut_paiement = 'reussi'
""")
ventes_federees.show(5)

RELEVES["vue federee (4 SGBD)"] = ventes_federees.count()
print("Lignes federees :", RELEVES["vue federee (4 SGBD)"])

+-----------+-------------+-----------+--------------------+------------+-----------------+------------+
|commande_id|          nom|      ville|             produit|   categorie|   moyen_paiement|montant_fcfa|
+-----------+-------------+-----------+--------------------+------------+-----------------+------------+
|      CMD01|     Awa Diop|      Dakar|Smartphone Tecno ...|  Telephonie|     Orange Money|       85000|
|      CMD02|Moussa Ndiaye|      Thies|Ordinateur portab...|Informatique|             Wave|      350000|
|      CMD03|   Fatou Sarr|      Dakar|         Parfum Musc|      Beaute|especes_livraison|       24000|
|      CMD04|Ibrahima Fall|Saint-Louis|        Boubou brode|        Mode|       Free Money|       45000|
|      CMD05|     Awa Diop|      Dakar|Casque Bluetooth JBL|  Telephonie|     Orange Money|       30000|
+-----------+-------------+-----------+--------------------+------------+-----------------+------------+
only showing top 5 rows

Lignes federees : 22


> ✍️ **Réflexion (exercice 5-6).** Pourquoi passe-t-on de 30 commandes à
> 29, puis 28, puis 22 lignes au fil des jointures internes ?
>
> *Votre réponse :Le principe est qu'une jointure interne ne garde que les lignes dont la clé existe dans les deux tables. Donc, s'il y'a l'id d'un produit, d'un client ou d'une commande qui ne figure pas dans l'autre table, le nombre de lignes diminue donc du nombre d'occurences concernées.
> En passant de 30 à 29 (jointure avec clients) : une commande porte un client_id absent de la table des clients ;
> En passant de 29 à 28 (jointure avec produits) : une commande porte un produit_id absent du catalogue ;
> En passant de 28 à 22 (jointure avec paiements) : six commandes n’ont aucun paiement enregistré.

---
# 7. Audit d'intégrité inter-SGBD

## 💡 Ce qu'il faut comprendre

Puisqu'aucune clé étrangère ne traverse les SGBD, **c'est à l'ingénieur
data de vérifier l'intégrité référentielle**. L'outil : l'**anti-jointure**
(`LEFT ANTI JOIN`), déjà croisée au TP 3, qui renvoie les lignes de gauche
**sans** correspondance à droite.

**Exercice 7.** Trouvez :
1. les commandes dont le client n'existe pas dans MySQL ;
2. les commandes dont le produit n'existe pas dans PostgreSQL ;
3. les paiements dont la commande n'existe pas dans Oracle.

In [73]:
# === À COMPLÉTER ===
cmd_sans_client = spark.sql("""
    SELECT co.* FROM commandes co
    LEFT ANTI JOIN clients cl ON co.client_id = cl.client_id
""")
cmd_sans_client.show()

cmd_sans_produit = spark.sql("""
    SELECT co.* FROM commandes co
    LEFT ANTI JOIN produits pr ON co.produit_id = pr.produit_id
""")
cmd_sans_produit.show()

paiements_orphelins = spark.sql("""
    SELECT co.* FROM commandes co
    LEFT ANTI JOIN paiements pa ON pa.commande_id = co.commande_id
""")
paiements_orphelins.show()

RELEVES["commandes sans client"]  = cmd_sans_client.count()
RELEVES["commandes sans produit"] = cmd_sans_produit.count()
RELEVES["paiements orphelins"]    = paiements_orphelins.count()

+-----------+---------+----------+--------+------------+-------------------+------+
|COMMANDE_ID|CLIENT_ID|PRODUIT_ID|QUANTITE|MONTANT_FCFA|      DATE_COMMANDE|STATUT|
+-----------+---------+----------+--------+------------+-------------------+------+
|      CMD21|     C999|      P002|       1|       15000|2026-02-24 00:00:00|livree|
+-----------+---------+----------+--------+------------+-------------------+------+

+-----------+---------+----------+--------+------------+-------------------+------+
|COMMANDE_ID|CLIENT_ID|PRODUIT_ID|QUANTITE|MONTANT_FCFA|      DATE_COMMANDE|STATUT|
+-----------+---------+----------+--------+------------+-------------------+------+
|      CMD22|     C009|      P999|       1|       50000|2026-02-26 00:00:00|livree|
+-----------+---------+----------+--------+------------+-------------------+------+

+-----------+---------+----------+--------+------------+-------------------+--------+
|COMMANDE_ID|CLIENT_ID|PRODUIT_ID|QUANTITE|MONTANT_FCFA|      DATE_COMMA

**Exercice 8 (jointure externe).** La Finance veut la liste des
commandes **livrées mais sans paiement réussi** (candidates à la relance).
Indice : `LEFT JOIN` + filtre sur les `NULL`, ou `LEFT ANTI JOIN` sur les
paiements réussis uniquement.

In [75]:
# === À COMPLÉTER ===
livrees_impayees = spark.sql("""
    SELECT co.commande_id, co.client_id, co.montant_fcfa
    FROM commandes co
    LEFT JOIN paiements pa
           ON pa.commande_id = co.commande_id
          AND pa.statut_paiement = 'reussi'
    WHERE co.statut = 'livree'
      AND pa.paiement_id IS NULL
    ORDER BY co.date_commande
""")
livrees_impayees.show()

RELEVES["livrees sans paiement reussi"] = livrees_impayees.count()

+-----------+---------+------------+
|commande_id|client_id|montant_fcfa|
+-----------+---------+------------+
+-----------+---------+------------+



---
# 8. Les analyses consolidées pour la direction

La vue fédérée en place, les questions métier deviennent du SQL ordinaire.

**Exercice 9.** Calculez, à partir de `ventes_federees` (exposée en vue
temporaire ci-dessous) :
1. le **CA total fédéré** ;
2. le CA **par ville**, trié décroissant ;
3. le CA **par catégorie** de produit, trié décroissant ;
4. le CA **par moyen de paiement**, trié décroissant.

In [83]:
ventes_federees.createOrReplaceTempView("ventes_federees")

# === À COMPLÉTER ===
ca_total = spark.sql("SELECT SUM(montant_fcfa) AS ca_total FROM ventes_federees")
ca_total.show()

ca_ville = spark.sql("""
    SELECT ville, SUM(montant_fcfa) AS ca_ville FROM ventes_federees
    GROUP BY ville 
    ORDER BY ca_ville DESC
""")
ca_ville.show()

ca_categorie = spark.sql("""
    SELECT categorie, SUM(montant_fcfa) AS ca_categorie
    FROM ventes_federees
    GROUP BY categorie
    ORDER BY ca_categorie DESC
""")
ca_categorie.show()

ca_moyen = spark.sql("""
    SELECT moyen_paiement, SUM(montant_fcfa) AS ca_moyen
    FROM ventes_federees
    GROUP BY moyen_paiement
    ORDER BY ca_moyen DESC
""")
ca_moyen.show()

RELEVES["CA federe total (FCFA)"] = ca_total.first()["ca_total"]

+--------+
|ca_total|
+--------+
| 1452000|
+--------+

+-----------+--------+
|      ville|ca_ville|
+-----------+--------+
|      Dakar|  723000|
|      Thies|  362000|
| Ziguinchor|  103000|
|   Rufisque|   85000|
|Saint-Louis|   60000|
|      Mbour|   46000|
|    Kaolack|   39000|
|      Touba|   28000|
|   Diourbel|    6000|
+-----------+--------+

+--------------+------------+
|     categorie|ca_categorie|
+--------------+------------+
|  Informatique|      740000|
|    Telephonie|      400000|
|          Mode|      174000|
|        Beaute|       84000|
|Electromenager|       54000|
+--------------+------------+

+-----------------+--------+
|   moyen_paiement|ca_moyen|
+-----------------+--------+
|             Wave|  537000|
|   carte_bancaire|  395000|
|     Orange Money|  371000|
|especes_livraison|   76000|
|       Free Money|   73000|
+-----------------+--------+



---
# 9. Sous le capot : que fait vraiment Spark ?

## 💡 Ce qu'il faut comprendre

Spark ne rapatrie pas aveuglément les 4 tables : quand c'est possible, il
**pousse les filtres et les projections vers les SGBD** (*predicate /
projection pushdown*), qui font le tri chez eux. Vérifions sur un filtre
simple, comme au TP 3 — mais cette fois sur Oracle :

In [84]:
livrees_dakar = (lire_jdbc(ORACLE_URL, "spark_user.commandes", ORACLE_PILOTE)
                 .filter(F.col("STATUT") == "livree")
                 .select("COMMANDE_ID", "MONTANT_FCFA"))

livrees_dakar.explain()
# Chercher dans le plan : PushedFilters: [*IsNotNull(STATUT),
# *EqualTo(STATUT,livree)] -> c'est ORACLE qui filtre, pas Spark.
# De meme, seules 2 colonnes traversent le reseau (projection pushdown).

== Physical Plan ==
*(1) Scan JDBCRelation(spark_user.commandes) [numPartitions=1] [COMMANDE_ID#1326,MONTANT_FCFA#1330] PushedFilters: [*IsNotNull(STATUT), *EqualTo(STATUT,livree)], ReadSchema: struct<COMMANDE_ID:string,MONTANT_FCFA:decimal(12,0)>




> ✍️ **Réflexion.** En production, la table `commandes` ferait des
> centaines de millions de lignes. Expliquez en 2 phrases pourquoi le
> pushdown change tout, et ce qui traverserait le réseau sans lui.
>
> *Votre réponse :* …

---
# 10. Boucler la boucle : écrire dans l'entrepôt

Lecture fédérée → analyses → **écriture du résultat consolidé**.
PostgreSQL joue ici le rôle d'entrepôt analytique : le script d'init a créé
un schéma `analytics` où `spark_user` a le droit d'écrire.

`mode("overwrite")` : le TP est rejouable à volonté.

In [85]:
(ventes_federees.write
    .format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", "analytics.ventes_federees")
    .option("user", UTILISATEUR)
    .option("password", MDP)
    .option("driver", PG_PILOTE)
    .mode("overwrite")
    .save())

print("Table analytics.ventes_federees ecrite dans PostgreSQL.")

Table analytics.ventes_federees ecrite dans PostgreSQL.


In [86]:
# Preuve : on relit ce qu'on vient d'ecrire
relecture = lire_jdbc(PG_URL, "analytics.ventes_federees", PG_PILOTE)
print("Lignes relues :", relecture.count())
relecture.show(5)
# Observation attendue : 22 lignes -- la boucle est bouclee : 4 SGBD en
# entree, 1 table consolidee en sortie, retrouvable dans pgAdmin/psql

Lignes relues : 22
+-----------+-------------+-----------+--------------------+------------+-----------------+------------+
|commande_id|          nom|      ville|             produit|   categorie|   moyen_paiement|montant_fcfa|
+-----------+-------------+-----------+--------------------+------------+-----------------+------------+
|      CMD01|     Awa Diop|      Dakar|Smartphone Tecno ...|  Telephonie|     Orange Money|       85000|
|      CMD02|Moussa Ndiaye|      Thies|Ordinateur portab...|Informatique|             Wave|      350000|
|      CMD03|   Fatou Sarr|      Dakar|         Parfum Musc|      Beaute|especes_livraison|       24000|
|      CMD04|Ibrahima Fall|Saint-Louis|        Boubou brode|        Mode|       Free Money|       45000|
|      CMD05|     Awa Diop|      Dakar|Casque Bluetooth JBL|  Telephonie|     Orange Money|       30000|
+-----------+-------------+-----------+--------------------+------------+-----------------+------------+
only showing top 5 rows



---
# 11. Tableau de relevés et clôture

In [87]:
print(f"{'Mesure':<38}{'Valeur':>12}")
print("-" * 50)
for k, v in RELEVES.items():
    print(f"{k:<38}{v:>12,}")

Mesure                                      Valeur
--------------------------------------------------
nb_clients (MySQL)                              12
nb_produits (PostgreSQL)                        10
nb_commandes (Oracle)                           30
nb_paiements (SQL Server)                       27
jointure commandes-clients                      29
jointure 3 tables                               28
vue federee (4 SGBD)                            22
commandes sans client                            1
commandes sans produit                           1
paiements orphelins                              4
livrees sans paiement reussi                     0
CA federe total (FCFA)                   1,452,000


In [88]:
spark.stop()
print("SparkSession arretee proprement.")

SparkSession arretee proprement.


---
# 12. Le livrable, en détail

À pousser sur votre dépôt GitHub **avant la séance 5** :

1. **Ce notebook complété et exécuté de bout en bout** (avec les sorties),
   dans `notebooks/TP4_federation_multi_sgbd.ipynb` ;
2. `docs/AUDIT_TP4.md` : les 3 anomalies d'intégrité trouvées en partie 7
   (identifiants, table source, SGBD, impact métier en 1 phrase chacune) ;
3. le tableau de relevés recopié dans `AUDIT_TP4.md`.

```bash
git add notebooks/TP4_federation_multi_sgbd.ipynb docs/AUDIT_TP4.md
git commit -m "TP4: federation 4 SGBD avec PySpark + audit d'integrite"
git push
```

## Questions de réflexion finales

1. La vue fédérée est **recalculée à chaque exécution** en interrogeant les
   4 SGBD de production. Quels risques en production, et quelle
   architecture du cours (séances 6-7) y répond ?

   *Votre réponse :* Cette architecture surchargent les différentes base de production notamment les transactions courantes (commandes, paiements). La lecture analytique consomme de la CPU, de la mémoire, des entrées/sorties et connexions.

   La disponibilité est fortement menacée par quatre points de défaillance sur le système (les SGBD). Un serveur indisponible affecte le disponibilité de la vue.

   les performances sont médiocres et imprévisible. D'une part, toutes les ligne traversent le réseau (traitement effectué par spark) et d'autres part, pour ce TP, il n'y a pas de parallélisme dans la lecture.

   Le risqued'incohérence des données est elevé. La lecture n'est pas englobée par une seul transaction alors que le déphasage entre les opérations est réel.Le paiement est effectué quelques secondes après la validation de la commande.

   Enfin, la surface d'attaque est élargie avec Spark qui détient les identifiants des quatre bases de données de production



   Cette architecture correspond à celle de ETL, le notebook n'a pas traité le chargement.

3. Citez un cas sénégalais réel où la fédération multi-SGBD serait
   nécessaire (pensez interconnexion d'administrations, banques/fintech,
   opérateurs).

   *Votre réponse :* Les demande de bourse ou d'aide sociale sont instruite en tenant compte :
 - l'état civil;
 - les inscriptions et les résultats de l'année ;
 - Admission dans la fonction publique
 - Les aides déjà rçues (le nombre d'années de validité de la bourse)
Ces systèmes sont tenus par des administrations différentes dont leur interconnexion serait nécessaire.